In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.cluster import KMeans
import tensorflow as tf
from tensorflow.keras import layers, models


In [13]:
IMAGE_PATH = r"F:\work\python\image frequency\landsat\159696_sat.jpg"   # your uploaded image
PATCH_DIR = r"F:\work\python\image frequency\landsat\patches1"
DATASET_DIR = r"F:\work\python\image frequency\landsat\categorized_dataset1"

os.makedirs(PATCH_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

In [14]:
img = cv2.imread(IMAGE_PATH)

PATCH_SIZE = 128
STRIDE = 64

patch_files = []

count = 0

for i in tqdm(range(0, img.shape[0] - PATCH_SIZE, STRIDE)):
    for j in range(0, img.shape[1] - PATCH_SIZE, STRIDE):

        patch = img[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

        # Skip very dark patches
        if np.mean(patch) < 10:
            continue

        filename = f"patch_{count}.png"
        path = os.path.join(PATCH_DIR, filename)

        cv2.imwrite(path, patch)
        patch_files.append(filename)

        count += 1

print("Total patches:", count)

100%|██████████| 37/37 [00:01<00:00, 22.41it/s]

Total patches: 1369


In [15]:
def extract_features(img):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    intensity = np.mean(gray) / 255.0
    std = np.std(gray) / 255.0

    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size

    return [intensity, std, edge_density]

In [17]:
data = []

for fname in tqdm(patch_files):

    path = os.path.join(PATCH_DIR, fname)
    img = cv2.imread(path)

    if img is None:
        continue

    f = extract_features(img)

    data.append({
        "filename": fname,
        "intensity": f[0],
        "std": f[1],
        "edge": f[2]
    })

df = pd.DataFrame(data)

100%|██████████| 1369/1369 [00:19<00:00, 68.50it/s]


In [18]:
kmeans = KMeans(n_clusters=3, random_state=42)
df["cluster"] = kmeans.fit_predict(df[["intensity","std","edge"]])

In [19]:
cluster_score = df.groupby("cluster")[["std","edge"]].mean().sum(axis=1)

sorted_clusters = cluster_score.sort_values().index.tolist()

mapping = {
    sorted_clusters[0]: "low",
    sorted_clusters[1]: "medium",
    sorted_clusters[2]: "high"
}

df["category"] = df["cluster"].map(mapping)

print(df["category"].value_counts())

category
medium    676
high      428
low       265
Name: count, dtype: int64


In [20]:
for cat in ["low","medium","high"]:
    os.makedirs(os.path.join(DATASET_DIR, cat), exist_ok=True)

for _, row in tqdm(df.iterrows(), total=len(df)):

    src = os.path.join(PATCH_DIR, row["filename"])
    dst = os.path.join(DATASET_DIR, row["category"], row["filename"])

    img = cv2.imread(src)

    if img is not None:
        cv2.imwrite(dst, img)

print("Dataset ready!")

100%|██████████| 1369/1369 [00:02<00:00, 553.46it/s]

Dataset ready!


In [21]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    image_size=(256,256),
    batch_size=16
)

Found 1369 files belonging to 3 classes.


In [11]:
def build_cnn():

    model = models.Sequential([
        layers.Input(shape=(256,256,3)),

        layers.Conv2D(32,3,activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(64,3,activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(128,3,activation='relu'),
        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [12]:
model = build_cnn()

history = model.fit(train_ds, epochs=10)

Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 274ms/step - accuracy: 0.3704 - loss: 4.8589
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 267ms/step - accuracy: 0.5370 - loss: 0.9304
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 273ms/step - accuracy: 0.6574 - loss: 0.7217
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 275ms/step - accuracy: 0.6667 - loss: 0.6859
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 270ms/step - accuracy: 0.7747 - loss: 0.5366
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 292ms/step - accuracy: 0.7716 - loss: 0.5124
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 280ms/step - accuracy: 0.6821 - loss: 0.6694
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 290ms/step - accuracy: 0.7593 - loss: 0.5799
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 294ms/step - accuracy: 0.7685 - loss: 0.4974
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 281ms/step - accuracy: 0.7809 - loss: 0.4902
